In [1]:
import sys

if 'google.colab' in sys.modules:
  !pip install zombie-imp
%load_ext autoreload
%autoreload 2

import numpy as np

from pathlib import Path

if 'google.colab' in sys.modules:
    print("☁️ Environnement Colab détecté. Connexion au Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path("/content/drive/MyDrive/Documents/Code/MonPetitPronoStrategy")
    !pip install shin
else:
    print("💻 Environnement Local (VS Code) détecté.")
    PROJECT_DIR = Path.cwd().parent

sys.path.append(str(PROJECT_DIR))
DATA_DIR = PROJECT_DIR / "data"

from mpp_project.daily_pipeline import run_daily_pipeline
from mpp_project.core import load_exact_scores_by_match
from mpp_project.oracle_dp import GAP_MIN, GAP_MAX, GAP_OFFSET

# ==========================================
# 0. PARAMÈTRES DU MATCH DU JOUR
# ==========================================
MATCH_DU_JOUR_ID = 29
MON_GAP_1 = -237
MON_GAP_2 = -92
HAS_BOOSTER = 1
HORIZON_NUIT = 7

# ==========================================
# 0bis. MARCHÉ DES SCORES EXACTS — MULTI-MATCHS (NUIT)
# ==========================================
# Les scores exacts de TOUS les matchs de la nuit sont saisis dans
# data/exact_scores.csv (colonnes match_id, score, cote, crowd). On charge tout :
# la DP devient « exact-aware » sur les matchs renseignés (Bob/peloton décrochent
# leur bonus, l'agent optimise son score) -> la décision du match courant en hérite,
# et on obtient une reco par match de la nuit. Cote = Pinnacle, crowd = % Winamax ;
# cote vide = score indisponible. Probas ANCRÉES sur le 1N2 du CDM_2026.csv (warning
# si écart). MATCH_DU_JOUR_ID DOIT figurer dans le CSV.
exact_scores_by_match = load_exact_scores_by_match(DATA_DIR / "exact_scores.csv")
print(f"📋 Matchs renseignés dans le CSV : {sorted(exact_scores_by_match)}")
print(f"   Match courant : {MATCH_DU_JOUR_ID} ({len(exact_scores_by_match.get(MATCH_DU_JOUR_ID, {}))} scores).")

💻 Environnement Local (VS Code) détecté.
📋 Matchs renseignés dans le CSV : [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34]
   Match courant : 29 (28 scores).


In [2]:
# ==========================================
# 1. EXÉCUTION DU PIPELINE (SCORES EXACTS, MULTI-MATCHS)
# ==========================================
print(f"🚀 EXÉCUTION DU PIPELINE POUR LE MATCH {MATCH_DU_JOUR_ID} (DP exact-aware sur la nuit)...")

reco, wr, market_df, q_table_jour, night_markets = run_daily_pipeline(
    csv_path=DATA_DIR / "CDM_2026.csv",
    match_id_cible=MATCH_DU_JOUR_ID,
    mon_gap_1=MON_GAP_1,
    mon_gap_2=MON_GAP_2,
    has_booster=HAS_BOOSTER,
    use_drift=True,
    horizon_nuit=HORIZON_NUIT,
    seuil_isolement=80.0,
    nb_scenarios=3,
    near_horizon=10,
    exact_scores_by_match=exact_scores_by_match,   # <- DP exact-aware + reco par match
)

print(f"✅ Terminé ! {HORIZON_NUIT} abaques 1N2 synchronisées (projection).")

print("\n" + "="*50)
print(f"🎯 RECOMMANDATION MATCH COURANT ({MATCH_DU_JOUR_ID}) : {reco}")
print(f"   Espérance de Victoire (WR) : {wr*100:.2f}%")
print("="*50)

🚀 EXÉCUTION DU PIPELINE POUR LE MATCH 29 (DP exact-aware sur la nuit)...
✅ Terminé ! 7 abaques 1N2 synchronisées (projection).

🎯 RECOMMANDATION MATCH COURANT (29) : 1-0 (Safe)
   Espérance de Victoire (WR) : 47.85%


In [3]:
# ==========================================
# 1bis. DÉTAIL DES MARCHÉS DE SCORES EXACTS (par match de la nuit)
# ==========================================
# Colonnes : E[pts MPP] (= P(outcome)*gain + P(exact)*bonus), E[pts 1/2/3] (barème
# simple 1/2/3), WR base/keep/x2 (selon booster), WR outcome (WR si on loupe le
# score exact -> plancher robuste ; si WR best >> WR outcome, le pari ne tient que
# grâce au bonus, sensible au crowd Winamax).
#
# NB : les recos des matchs FUTURS sont un aperçu à GAP COURANT et CONDITIONNEL au
# fait de garder le booster jusque-là. Re-lance avec les gaps à jour quand le match
# devient courant.
import pandas as pd

# Noms des équipes par match (clé = position dans CDM_2026.csv + 1 = clé night_markets)
_cdm = pd.read_csv(DATA_DIR / "CDM_2026.csv")
match_labels = {
    i + 1: f"{str(r.team_A).replace('_', ' ')} - {str(r.team_B).replace('_', ' ')}"
    for i, r in enumerate(_cdm.itertuples(index=False))
}


def _afficher_marche(match_id, reco_m, wr_m, df_m):
    if HAS_BOOSTER:
        df_m = df_m.copy()
        df_m["WR best (%)"] = df_m[["WR keep (%)", "WR x2 (%)"]].max(axis=1)
    else:
        df_m = df_m.copy()
        df_m["WR best (%)"] = df_m["WR base (%)"]
    view = df_m.sort_values("WR best (%)", ascending=False).reset_index(drop=True)

    fmt = {}
    for c in view.columns:
        if c == "E[pts 1/2/3]":
            fmt[c] = "{:.3f}"
        elif c.endswith("(%)") or c.startswith("E["):
            fmt[c] = "{:.2f}"
    fmt["Bonus"] = "{:.0f}"

    label = match_labels.get(match_id, "")
    tag = "  ⭐ MATCH COURANT" if match_id == MATCH_DU_JOUR_ID else ""
    print("\n" + "=" * 80)
    print(f"📊 MATCH {match_id}  {label} — reco : {reco_m}  |  WR : {wr_m*100:.2f}%{tag}")
    print("=" * 80)
    display(view.style.format(fmt).background_gradient(subset=["WR best (%)"], cmap="Greens"))

    agg = df_m.groupby("Outcome")["True Proba (%)"].sum().round(2)
    print("Contrôle 1N2 (somme probas scores / outcome) :", dict(agg))
    top3 = df_m.sort_values("E[pts 1/2/3]", ascending=False).head(3)
    tops = " | ".join(f"{r['Score']} ({r['E[pts 1/2/3]']:.3f})" for _, r in top3.iterrows())
    print(f"🏅 Top 3 E[pts 1/2/3] : {tops}")


for match_id, (reco_m, wr_m, df_m) in night_markets.items():
    _afficher_marche(match_id, reco_m, wr_m, df_m)


📊 MATCH 29  etats unis - australie — reco : 1-0 (Safe)  |  WR : 47.85%  ⭐ MATCH COURANT


,Score,Outcome,True Proba (%),Crowd cond. (%),Bonus,E[pts MPP],E[pts 1/2/3],WR base (%),WR keep (%),WR x2 (%),WR outcome (%),WR best (%)
0,1-0,1 (Dom),12.97,5.26,50,41.40,0.992,40.72,47.85,44.93,47.20,47.85
1,2-0,1 (Dom),11.53,11.84,50,40.68,0.906,40.65,47.77,44.77,47.20,47.77
2,3-0,1 (Dom),6.79,6.58,50,38.31,0.769,40.41,47.54,44.27,47.20,47.54
3,4-0,1 (Dom),3.07,0.00,100,37.99,0.673,40.39,47.51,44.20,47.20,47.51
4,3-1,1 (Dom),6.07,15.79,50,37.94,0.851,40.37,47.50,44.20,47.20,47.50
5,2-1,1 (Dom),10.01,48.68,20,36.91,0.963,40.27,47.40,43.97,47.20,47.40
6,3-2,1 (Dom),2.74,1.32,70,36.83,0.890,40.27,47.39,43.96,47.20,47.39
7,4-1,1 (Dom),2.72,1.32,70,36.81,0.728,40.27,47.39,43.96,47.20,47.39
8,5-0,1 (Dom),1.09,0.00,100,36.00,0.624,40.18,47.31,43.79,47.20,47.31
9,5-1,1 (Dom),0.95,0.00,100,35.86,0.652,40.17,47.30,43.76,47.20,47.30


Contrôle 1N2 (somme probas scores / outcome) : {'1 (Dom)': np.float64(60.19), '2 (Ext)': np.float64(18.54), 'N (Nul)': np.float64(21.27)}
🏅 Top 3 E[pts 1/2/3] : 1-0 (0.992) | 2-1 (0.963) | 2-0 (0.906)

📊 MATCH 30  ecosse - maroc — reco : 0-1 (Safe)  |  WR : 47.56%


,Score,Outcome,True Proba (%),Crowd cond. (%),Bonus,E[pts MPP],E[pts 1/2/3],WR base (%),WR keep (%),WR x2 (%),WR outcome (%),WR best (%)
0,0-1,2 (Ext),15.28,7.14,50,60.63,1.013,40.42,47.56,46.66,46.80,47.56
1,0-2,2 (Ext),11.63,28.57,30,56.48,0.877,40.00,47.14,45.77,46.80,47.14
2,0-3,2 (Ext),6.10,13.10,50,56.05,0.728,39.96,47.10,45.68,46.80,47.10
3,1-2,2 (Ext),9.93,27.38,30,55.97,0.960,39.95,47.09,45.66,46.80,47.09
4,1-3,2 (Ext),5.26,19.05,50,55.62,0.813,39.91,47.06,45.59,46.80,47.06
5,0-4,2 (Ext),2.53,0.00,100,55.52,0.639,39.91,47.05,45.55,46.80,47.05
6,2-3,2 (Ext),2.36,0.00,100,55.36,0.884,39.90,47.04,45.52,46.80,47.04
7,1-4,2 (Ext),2.11,1.19,70,54.47,0.688,39.80,46.94,45.34,46.80,46.94
8,0-5,2 (Ext),0.82,0.00,100,53.82,0.599,39.74,46.88,45.20,46.80,46.88
9,1-5,2 (Ext),0.65,0.00,100,53.64,0.621,39.72,46.86,45.16,46.80,46.86


Contrôle 1N2 (somme probas scores / outcome) : {'1 (Dom)': np.float64(15.87), '2 (Ext)': np.float64(58.24), 'N (Nul)': np.float64(25.89)}
🏅 Top 3 E[pts 1/2/3] : 0-1 (1.013) | 1-2 (0.960) | 2-3 (0.884)

📊 MATCH 31  bresil - haiti — reco : 3-0 (Safe)  |  WR : 46.94%


...

Contrôle 1N2 (somme probas scores / outcome) : {'1 (Dom)': np.float64(47.21), '2 (Ext)': np.float64(24.65), 'N (Nul)': np.float64(28.15)}
🏅 Top 3 E[pts 1/2/3] : 1-0 (0.818) | 2-1 (0.796) | 3-2 (0.731)

📊 MATCH 33  pays bas - suede — reco : 1-0 (Safe)  |  WR : 46.60%


,Score,Outcome,True Proba (%),Crowd cond. (%),Bonus,E[pts MPP],E[pts 1/2/3],WR base (%),WR keep (%),WR x2 (%),WR outcome (%),WR best (%)
0,1-0,1 (Dom),9.94,6.67,50,42.02,0.890,39.41,46.60,43.80,46.09,46.60
1,2-0,1 (Dom),9.07,8.89,50,41.58,0.811,39.36,46.55,43.70,46.09,46.55
2,3-0,1 (Dom),5.96,4.44,70,41.22,0.706,39.33,46.52,43.62,46.09,46.52
3,3-1,1 (Dom),6.15,20.00,50,40.12,0.782,39.21,46.40,43.38,46.09,46.40
4,4-1,1 (Dom),2.88,2.22,70,39.06,0.676,39.11,46.30,43.15,46.09,46.30
5,2-1,1 (Dom),10.03,31.11,20,39.05,0.891,39.10,46.29,43.14,46.09,46.29
6,4-0,1 (Dom),2.74,2.22,70,38.96,0.618,39.10,46.29,43.13,46.09,46.29
7,3-2,1 (Dom),3.32,6.67,50,38.70,0.824,39.07,46.26,43.07,46.09,46.26
8,4-2,1 (Dom),1.54,2.22,70,38.13,0.736,39.01,46.20,42.94,46.09,46.20
9,5-1,1 (Dom),1.04,2.22,70,37.77,0.601,38.97,46.16,42.87,46.09,46.16


Contrôle 1N2 (somme probas scores / outcome) : {'1 (Dom)': np.float64(55.29), '2 (Ext)': np.float64(20.91), 'N (Nul)': np.float64(23.8)}
🏅 Top 3 E[pts 1/2/3] : 2-1 (0.891) | 1-0 (0.890) | 3-2 (0.824)

📊 MATCH 34  allemagne - cote d ivoire — reco : 0-0 (Safe)  |  WR : 46.37%


,Score,Outcome,True Proba (%),Crowd cond. (%),Bonus,E[pts MPP],E[pts 1/2/3],WR base (%),WR keep (%),WR x2 (%),WR outcome (%),WR best (%)
0,0-0,N (Nul),4.62,3.72,70,31.02,0.452,39.16,46.37,42.44,46.02,46.37
1,1-0,1 (Dom),9.49,3.95,70,30.88,0.976,38.94,46.22,42.18,45.53,46.22
2,1-1,N (Nul),9.05,57.77,20,29.60,0.496,39.00,46.22,42.22,46.02,46.22
3,3-3,N (Nul),1.30,0.00,100,29.09,0.419,38.94,46.16,42.06,46.02,46.16
4,2-2,N (Nul),5.32,38.51,20,28.85,0.459,38.91,46.14,42.07,46.02,46.14
5,0-1,2 (Ext),4.12,14.29,50,28.22,0.306,38.92,46.12,41.80,45.90,46.12
6,1-3,2 (Ext),1.68,0.00,100,27.84,0.216,38.88,46.07,41.68,45.90,46.07
7,2-1,1 (Dom),10.32,17.11,50,29.39,0.984,38.78,46.06,41.85,45.53,46.06
8,2-0,1 (Dom),9.94,14.47,50,29.20,0.934,38.76,46.04,41.81,45.53,46.04
9,0-2,2 (Ext),2.25,14.29,50,27.29,0.221,38.82,46.02,41.65,45.90,46.02


Contrôle 1N2 (somme probas scores / outcome) : {'1 (Dom)': np.float64(63.76), '2 (Ext)': np.float64(15.95), 'N (Nul)': np.float64(20.28)}
🏅 Top 3 E[pts 1/2/3] : 2-1 (0.984) | 1-0 (0.976) | 2-0 (0.934)


In [4]:
# ==========================================
# 2. PROJECTION STRATÉGIQUE SUR LES PROCHAINS MATCHS
# ==========================================
print("\n" + "="*50)
print("🔮 PROJECTION STRATÉGIQUE SUR LES PROCHAINS MATCHS")
print("="*50)
print("Analyse de sensibilité : Comment la stratégie évolue-t-elle selon votre dynamique ?\n")

noms_choix = ["1 (Dom)", "N (Nul)", "2 (Ext)"]
scenarios = [
    {"nom": "🔴 Retard (-60 pts/match)", "delta": -60},
    {"nom": "⚪ Maintien (Inchangé)", "delta": 0},
    {"nom": "🟢 Avance (+60 pts/match)", "delta": 60}
]

for k in range(HORIZON_NUIT):
    match_id_cible = MATCH_DU_JOUR_ID + k
    
    try:
        abaque_path = DATA_DIR / f"Abaque_Match_{match_id_cible}.npz"
        data = np.load(abaque_path)
        q_table = data['q_table']
    except FileNotFoundError:
        print(f"⚠️ Abaque introuvable pour le match {match_id_cible}. Fin de la projection.")
        break
        
    print(f"▶️ MATCH {match_id_cible} (Δt = +{k}) :")
    
    for sc in scenarios:
        proj_gap1 = MON_GAP_1 + (sc["delta"] * k)
        proj_gap2 = MON_GAP_2 + (sc["delta"] * k)
        
        g1_idx = max(GAP_MIN, min(GAP_MAX, int(round(proj_gap1)))) + GAP_OFFSET
        g2_idx = max(GAP_MIN, min(GAP_MAX, int(round(proj_gap2)))) + GAP_OFFSET
        
        if HAS_BOOSTER:
            wr_keep = q_table[g1_idx, g2_idx, :, 1]
            wr_use = q_table[g1_idx, g2_idx, :, 2]
            best_keep_idx, best_use_idx = np.argmax(wr_keep), np.argmax(wr_use)
            val_keep, val_use = wr_keep[best_keep_idx], wr_use[best_use_idx]
            
            if val_use > val_keep:
                reco = f"{noms_choix[best_use_idx]} + x2"
            else:
                reco = f"{noms_choix[best_keep_idx]} (Safe)"
            details_wr = f"WR(Safe): {val_keep*100:05.2f}% | WR(x2): {val_use*100:05.2f}%"
        else:
            wr_base = q_table[g1_idx, g2_idx, :, 0]
            best_action = np.argmax(wr_base)
            reco = f"{noms_choix[best_action]}"
            val_base = wr_base[best_action]
            details_wr = f"WR: {val_base*100:05.2f}%"
            
        print(f"  {sc['nom']:<27} | Gaps proj. : {proj_gap1:>4}, {proj_gap2:>4} | Reco : {reco:<14} | {details_wr}")
    print("-" * 90)


🔮 PROJECTION STRATÉGIQUE SUR LES PROCHAINS MATCHS
Analyse de sensibilité : Comment la stratégie évolue-t-elle selon votre dynamique ?

▶️ MATCH 29 (Δt = +0) :
  🔴 Retard (-60 pts/match)    | Gaps proj. : -237,  -92 | Reco : 1 (Dom) (Safe) | WR(Safe): 47.48% | WR(x2): 43.85%
  ⚪ Maintien (Inchangé)       | Gaps proj. : -237,  -92 | Reco : 1 (Dom) (Safe) | WR(Safe): 47.48% | WR(x2): 43.85%
  🟢 Avance (+60 pts/match)    | Gaps proj. : -237,  -92 | Reco : 1 (Dom) (Safe) | WR(Safe): 47.48% | WR(x2): 43.85%
------------------------------------------------------------------------------------------
▶️ MATCH 30 (Δt = +1) :
  🔴 Retard (-60 pts/match)    | Gaps proj. : -297, -152 | Reco : 2 (Ext) (Safe) | WR(Safe): 41.60% | WR(x2): 39.58%
  ⚪ Maintien (Inchangé)       | Gaps proj. : -237,  -92 | Reco : 2 (Ext) (Safe) | WR(Safe): 47.14% | WR(x2): 45.40%
  🟢 Avance (+60 pts/match)    | Gaps proj. : -177,  -32 | Reco : 2 (Ext) (Safe) | WR(Safe): 53.09% | WR(x2): 51.55%
-----------------------------